In [1]:

import json
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np


# ============================================================
# Configuration
# ============================================================

RESULTS_DIR = Path("../results")
PLOTS_DIR = RESULTS_DIR / "plots"

PLOTS_DIR.mkdir(parents=True, exist_ok=True)


# ============================================================
# Helper function: load JSON
# ============================================================

def load_json(path):
    with open(path, "r") as f:
        return json.load(f)


# ============================================================
# 1. Load all test-result JSON files
# ============================================================

json_files = list(RESULTS_DIR.glob("*.json"))

if not json_files:
    raise FileNotFoundError(
        f"No JSON files found in {RESULTS_DIR.resolve()}"
    )

results = {}

for path in json_files:
    data = load_json(path)

    model_path = data.get("model", "")

    results[path.name] = data

    print(
        f"Loaded: {path.name} "
        f"| Accuracy: {data['accuracy'] * 100:.2f}% "
        f"| Macro F1: {data['macro_f1'] * 100:.2f}%"
    )


# ============================================================
# Helper: identify model type and temperature
# ============================================================

def identify_result(filename, data):
    filename_lower = filename.lower()
    model_path = data.get("model", "").lower()

    combined = filename_lower + " " + model_path

    # Temperature-specific KD models
    if "kd2" in combined:
        return "KD T=2", 2

    if "kd4" in combined:
        return "KD T=4", 4

    if "kd6" in combined:
        return "KD T=6", 6

    # Baseline BiLSTM
    if "bilstm_student" in combined:
        return "BiLSTM Baseline", None

    if "baseline" in combined:
        return "BiLSTM Baseline", None

    # BERT teacher
    if "bert_teacher" in combined:
        return "BERT Teacher", None

    return None, None


identified_results = []

for filename, data in results.items():

    label, temperature = identify_result(filename, data)

    if label is not None:
        identified_results.append(
            {
                "label": label,
                "temperature": temperature,
                "data": data,
                "filename": filename,
            }
        )


# ============================================================
# Print discovered models
# ============================================================

print("\nDiscovered models:")

for item in identified_results:
    print(
        f"  {item['label']} "
        f"| {item['filename']}"
    )


# ============================================================
# 2. Temperature comparison
# ============================================================

temperature_results = [
    item
    for item in identified_results
    if item["temperature"] is not None
]

temperature_results.sort(
    key=lambda x: x["temperature"]
)

if temperature_results:

    temperatures = [
        item["temperature"]
        for item in temperature_results
    ]

    accuracies = [
        item["data"]["accuracy"] * 100
        for item in temperature_results
    ]

    macro_f1 = [
        item["data"]["macro_f1"] * 100
        for item in temperature_results
    ]

    fig, ax = plt.subplots(figsize=(8, 5))

    ax.plot(
        temperatures,
        accuracies,
        marker="o",
        linewidth=2,
        label="Accuracy",
    )

    ax.plot(
        temperatures,
        macro_f1,
        marker="o",
        linewidth=2,
        label="Macro F1",
    )

    ax.set_xlabel("Temperature (T)")
    ax.set_ylabel("Score (%)")

    

    ax.set_xticks(temperatures)

    ax.grid(
        True,
        alpha=0.3,
    )

    ax.legend()

    plt.tight_layout()

    output_path = (
        PLOTS_DIR /
        "temperature_comparison.png"
    )

    plt.savefig(
        output_path,
        dpi=300,
        bbox_inches="tight",
    )

    plt.close()

    print(
        f"\nSaved: {output_path}"
    )


# ============================================================
# 3. Overall model comparison
# ============================================================

model_order = [
    "BERT Teacher",
    "BiLSTM Baseline",
    "KD T=2",
    "KD T=4",
    "KD T=6",
]

ordered_models = []

for model_name in model_order:

    matches = [
        item
        for item in identified_results
        if item["label"] == model_name
    ]

    if matches:
        ordered_models.append(matches[0])


if ordered_models:

    labels = [
        item["label"]
        for item in ordered_models
    ]

    accuracies = [
        item["data"]["accuracy"] * 100
        for item in ordered_models
    ]

    macro_f1 = [
        item["data"]["macro_f1"] * 100
        for item in ordered_models
    ]

    x = np.arange(len(labels))

    width = 0.35

    fig, ax = plt.subplots(
        figsize=(10, 6)
    )

    bars1 = ax.bar(
        x - width / 2,
        accuracies,
        width,
        label="Accuracy",
    )

    bars2 = ax.bar(
        x + width / 2,
        macro_f1,
        width,
        label="Macro F1",
    )

    ax.set_xlabel("Model")

    ax.set_ylabel("Score (%)")

    ax.set_title(
        "Overall Model Performance"
    )

    ax.set_xticks(x)

    ax.set_xticklabels(
        labels,
        rotation=20,
        ha="right",
    )

    ax.legend()

    ax.grid(
        axis="y",
        alpha=0.3,
    )

    # Add values above bars
    for bars in [bars1, bars2]:

        for bar in bars:

            height = bar.get_height()

            ax.annotate(
                f"{height:.2f}",
                xy=(
                    bar.get_x() + bar.get_width() / 2,
                    height,
                ),
                xytext=(0, 3),
                textcoords="offset points",
                ha="center",
                va="bottom",
                fontsize=9,
            )

    plt.tight_layout()

    output_path = (
        PLOTS_DIR /
        "model_comparison.png"
    )

    plt.savefig(
        output_path,
        dpi=300,
        bbox_inches="tight",
    )

    plt.close()

    print(
        f"Saved: {output_path}"
    )


# ============================================================
# 4. Per-class F1 comparison
# ============================================================

class_names = [
    "World",
    "Sports",
    "Business",
    "Sci/Tech",
]

class_models = [
    "BERT Teacher",
    "BiLSTM Baseline",
    "KD T=2",
    "KD T=4",
    "KD T=6",
]

class_results = []

for model_name in class_models:

    matches = [
        item
        for item in identified_results
        if item["label"] == model_name
    ]

    if matches:
        class_results.append(matches[0])


if class_results:

    x = np.arange(len(class_names))

    width = (
        0.8 / len(class_results)
    )

    fig, ax = plt.subplots(
        figsize=(11, 6)
    )

    for i, item in enumerate(class_results):

        values = [
            item["data"]["per_class"][class_name]["f1"] * 100
            for class_name in class_names
        ]

        offset = (
            i - (len(class_results) - 1) / 2
        ) * width

        bars = ax.bar(
            x + offset,
            values,
            width,
            label=item["label"],
        )

        # Add values above bars
        for bar in bars:

            height = bar.get_height()

            ax.annotate(
                f"{height:.1f}",
                xy=(
                    bar.get_x() + bar.get_width() / 2,
                    height,
                ),
                xytext=(0, 2),
                textcoords="offset points",
                ha="center",
                va="bottom",
                fontsize=7,
            )

    ax.set_xlabel("Class")

    ax.set_ylabel("F1 Score (%)")

    ax.set_title(
        "Per-Class F1 Score Comparison"
    )

    ax.set_xticks(x)

    ax.set_xticklabels(class_names)

    ax.legend()

    ax.grid(
        axis="y",
        alpha=0.3,
    )

    plt.tight_layout()

    output_path = (
        PLOTS_DIR /
        "per_class_f1_comparison.png"
    )

    plt.savefig(
        output_path,
        dpi=300,
        bbox_inches="tight",
    )

    plt.close()

    print(
        f"Saved: {output_path}"
    )


# ============================================================
# Finished
# ============================================================

print(
    "\nAll available plots have been generated."
)

print(
    f"Plots are located in: {PLOTS_DIR.resolve()}"
)



Loaded: bert_test_results.json | Accuracy: 94.41% | Macro F1: 94.40%
Loaded: bilstm_kd6_test_results.json | Accuracy: 91.97% | Macro F1: 91.95%
Loaded: bilstm_kd2_test_results.json | Accuracy: 92.63% | Macro F1: 92.62%
Loaded: bilstm_kd4_test_results.json | Accuracy: 92.09% | Macro F1: 92.10%
Loaded: bilstm_baseline_results.json | Accuracy: 90.86% | Macro F1: 90.81%

Discovered models:
  BERT Teacher | bert_test_results.json
  KD T=6 | bilstm_kd6_test_results.json
  KD T=2 | bilstm_kd2_test_results.json
  KD T=4 | bilstm_kd4_test_results.json
  BiLSTM Baseline | bilstm_baseline_results.json

Saved: ../results/plots/temperature_comparison.png
Saved: ../results/plots/model_comparison.png
Saved: ../results/plots/per_class_f1_comparison.png

All available plots have been generated.
Plots are located in: /Users/harshithahonnappa/KnowledgeDistillation/results/plots
